In [11]:
from pathlib import Path

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/googleplaystore.csv")
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [12]:
if "Reviews" in df.columns:
    original = df["Reviews"].copy()
    cleaned = original.astype("string").str.strip().str.replace(",", "", regex=False)
    cleaned = cleaned.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    converted = pd.to_numeric(cleaned, errors="coerce")

    failed = original.notna() & cleaned.notna() & converted.isna()
    print("Reviews values that could not be converted:", int(failed.sum()))
    if failed.any():
        print(original[failed].drop_duplicates().head(20))

    # Review counts should be whole numbers and non-negative.
    invalid = converted.notna() & ((converted < 0) | (converted % 1 != 0))
    print("Negative or non-integer review counts:", int(invalid.sum()))
    converted.loc[invalid] = np.nan

    df["Reviews"] = converted.astype("Int64")
    print(df["Reviews"].describe())
else:
    print("Column 'Reviews' not found; skipped.")


Reviews values that could not be converted: 1
10472    3.0M
Name: Reviews, dtype: str
Negative or non-integer review counts: 0
count           10840.0
mean      444152.896033
std      2927760.603886
min                 0.0
25%                38.0
50%              2094.0
75%             54775.5
max          78158306.0
Name: Reviews, dtype: Float64


In [13]:
if "Installs" in df.columns:
    original = df["Installs"].copy()
    cleaned = (
        original.astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
    )
    cleaned = cleaned.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    converted = pd.to_numeric(cleaned, errors="coerce")

    failed = original.notna() & cleaned.notna() & converted.isna()
    print("Installs values that could not be converted:", int(failed.sum()))
    if failed.any():
        print(original[failed].drop_duplicates().head(20))

    invalid = converted.notna() & ((converted < 0) | (converted % 1 != 0))
    print("Negative or non-integer install counts:", int(invalid.sum()))
    converted.loc[invalid] = np.nan

    df["Installs"] = converted.astype("Int64")
    print(df["Installs"].describe())
else:
    print("Column 'Installs' not found; skipped.")


Installs values that could not be converted: 1
10472    Free
Name: Installs, dtype: str
Negative or non-integer install counts: 0
count            10840.0
mean     15464338.882565
std      85029361.395463
min                  0.0
25%               1000.0
50%             100000.0
75%            5000000.0
max         1000000000.0
Name: Installs, dtype: Float64


In [14]:
if "Price" in df.columns:
    original = df["Price"].copy()
    cleaned = (
        original.astype("string")
        .str.strip()
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    cleaned = cleaned.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    converted = pd.to_numeric(cleaned, errors="coerce")

    failed = original.notna() & cleaned.notna() & converted.isna()
    print("Price values that could not be converted:", int(failed.sum()))
    if failed.any():
        print(original[failed].drop_duplicates().head(20))

    invalid = converted.notna() & (converted < 0)
    print("Negative prices:", int(invalid.sum()))
    converted.loc[invalid] = np.nan

    df["Price_USD"] = converted.round(2)
    df = df.drop(columns=["Price"])
    print(df["Price_USD"].describe())
else:
    print("Column 'Price' not found; skipped.")


Price values that could not be converted: 1
10472    Everyone
Name: Price, dtype: str
Negative prices: 0
count      10840.0
mean      1.027368
std      15.949703
min            0.0
25%            0.0
50%            0.0
75%            0.0
max          400.0
Name: Price_USD, dtype: Float64


In [15]:
import re

def size_to_mb(value):
    """Convert a Play Store size string to MB; return NaN if invalid."""

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if text in {"", "nan", "none", "varies with device"}:
        return np.nan

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*([mk])", text)

    if not match:
        return np.nan

    amount = float(match.group(1))
    unit = match.group(2)

    if amount < 0:
        return np.nan

    if unit == "m":
        return amount
    else:
        return amount / 1000


if "Size" in df.columns:
    original = df["Size"].copy()

    df["Size_MB"] = original.map(size_to_mb)

    unparsed = original.notna() & df["Size_MB"].isna()

    print("Non-missing Size values that could not be converted:",
          int(unparsed.sum()))

    if unparsed.any():
        print(original[unparsed].value_counts().head(20))

    df.drop(columns=["Size"], inplace=True)

    print(df["Size_MB"].describe())

else:
    print("Column 'Size' not found; skipped.")

Non-missing Size values that could not be converted: 1696
Size
Varies with device    1695
1,000+                   1
Name: count, dtype: int64
count    9145.000000
mean       21.516530
std        22.588748
min         0.008500
25%         4.900000
50%        13.000000
75%        30.000000
max       100.000000
Name: Size_MB, dtype: float64


In [16]:
if "Last Updated" in df.columns:
    original = df["Last Updated"].copy()
    converted = pd.to_datetime(original, errors="coerce")

    failed = original.notna() & converted.isna()
    print("Last Updated values that could not be parsed:", int(failed.sum()))
    if failed.any():
        print(original[failed].drop_duplicates().head(20))

    df["Last Updated"] = converted
    print("Date range:")
    print(df["Last Updated"].min(), "to", df["Last Updated"].max())
else:
    print("Column 'Last Updated' not found; skipped.")


Last Updated values that could not be parsed: 1
10472    1.0.19
Name: Last Updated, dtype: str
Date range:
2010-05-21 00:00:00 to 2018-08-08 00:00:00


In [17]:
for column in ["Category", "Type", "Content Rating", "Genres"]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()
        df[column] = df[column].replace("", pd.NA)

# Standardize Type labels if present.
if "Type" in df.columns:
    df["Type"] = df["Type"].str.title()

for column in ["Category", "Type", "Content Rating", "Genres"]:
    if column in df.columns:
        print(f"\\n{column} — most frequent values")
        print(df[column].value_counts(dropna=False).head(15))


\nCategory — most frequent values
Category
FAMILY                1972
GAME                  1144
TOOLS                  843
MEDICAL                463
BUSINESS               460
PRODUCTIVITY           424
PERSONALIZATION        392
COMMUNICATION          387
SPORTS                 384
LIFESTYLE              382
FINANCE                366
HEALTH_AND_FITNESS     341
PHOTOGRAPHY            335
SOCIAL                 295
NEWS_AND_MAGAZINES     283
Name: count, dtype: Int64
\nType — most frequent values
Type
Free    10039
Paid      800
<NA>        1
0           1
Name: count, dtype: Int64
\nContent Rating — most frequent values
Content Rating
Everyone           8714
Teen               1208
Mature 17+          499
Everyone 10+        414
Adults only 18+       3
Unrated               2
<NA>                  1
Name: count, dtype: Int64
\nGenres — most frequent values
Genres
Tools               842
Entertainment       623
Education           549
Medical             463
Business            460
P

In [18]:
print(df.dtypes.to_frame("Final dtype"))

validation = {}

if "Reviews" in df.columns:
    validation["negative_reviews"] = int((df["Reviews"].dropna() < 0).sum())

if "Installs" in df.columns:
    validation["negative_installs"] = int((df["Installs"].dropna() < 0).sum())

if "Price_USD" in df.columns:
    validation["negative_prices"] = int((df["Price_USD"].dropna() < 0).sum())

if "Size_MB" in df.columns:
    validation["negative_sizes"] = int((df["Size_MB"].dropna() < 0).sum())

if "Rating" in df.columns:
    validation["ratings_outside_1_to_5"] = int(
        (df["Rating"].notna() & ~df["Rating"].between(1, 5)).sum()
    )

print("Validation checks:", validation)
print("Remaining exact duplicate rows:", int(df.duplicated().sum()))
print("Total missing cells:", int(df.isna().sum().sum()))


                   Final dtype
App                        str
Category                string
Rating                 float64
Reviews                  Int64
Installs                 Int64
Type                    string
Content Rating          string
Genres                  string
Last Updated    datetime64[us]
Current Ver                str
Android Ver                str
Price_USD              Float64
Size_MB                float64
Validation checks: {'negative_reviews': 0, 'negative_installs': 0, 'negative_prices': 0, 'negative_sizes': 0, 'ratings_outside_1_to_5': 1}
Remaining exact duplicate rows: 483
Total missing cells: 3187


In [19]:
OUTPUT_PATH = Path("../data/processed/googleplaystore_preprocessed.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Preprocessed dataset saved to: {OUTPUT_PATH.resolve()}")
print("Final shape:", df.shape)
display(df.head())


Preprocessed dataset saved to: C:\Github_Projects\Google_AppStore_EDA\data\processed\googleplaystore_preprocessed.csv
Final shape: (10841, 13)


,App,Category,Rating,Reviews,Installs,Type,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Price_USD,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,10000,Free,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,0.0,19.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967,500000,Free,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,0.0,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,5000000,Free,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,0.0,8.7
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,50000000,Free,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,0.0,25.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,100000,Free,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,0.0,2.8


Data preprocessing Summary

Transformations performed
- Converted Reviews and Installs to nullable integer counts.
- Converted Price to numeric Price_USD.
- Converted parseable Size values to Size_MB.
- Parsed Last Updated into datetime.
- Standardized whitespace and selected categorical labels.
- Reported conversion failures and validated numeric constraints.

Important limitations
- Installs is a displayed threshold from the source, not necessarily an exact count.
- Size_MB is missing for values such as Varies with device.
- No arbitrary imputation or outlier deletion was performed.
- No scaling or one-hot encoding was applied because the next notebook is descriptive EDA, not model training.